# 05 — Observability and Langfuse Tracing

memtomem-stm provides three layers of observability:

1. **MCP tools** — `stm_proxy_stats`, `stm_proxy_health`,
   `stm_surfacing_stats` give you an agent-accessible summary
   of what the proxy is doing.
2. **SQLite metrics** — `proxy_metrics.db` and
   `stm_feedback.db` persist per-call metrics for offline
   analysis.
3. **Langfuse tracing** — optional nested spans that let you
   visualize the full proxy pipeline in a shared team UI.

This notebook demonstrates all three layers. Core cells run
without any API keys; the live Langfuse cells gate themselves
on `_HAS_LANGFUSE_KEY`.

**Prereqs:** `uv sync` (dev group). Optional:
`pip install "memtomem-stm[langfuse]"` plus Langfuse
credentials for live tracing.

## 1. Isolate state and import helpers

In [ ]:
# Add notebooks/ to sys.path so `_helpers` is importable regardless of
# where Jupyter was launched (from the repo root or from notebooks/).
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd))
elif (_cwd / "notebooks" / "_helpers.py").exists():
    sys.path.insert(0, str(_cwd / "notebooks"))
else:
    raise RuntimeError(
        "Cannot find notebooks/_helpers.py — run Jupyter from the repo "
        "root or from the notebooks/ directory."
    )

In [ ]:
import os
import subprocess

from _helpers import (
    isolate_stm_state,
    stm_session,
    extract_text,
    fixtures_dir,
)

config_path = isolate_stm_state(prefix="nb05_")
print(f"STM config → {config_path}")

## 2. Register a fixture server and make a call

We need proxy calls to generate stats. Let's add the echo
fixture and make a quick call.

In [ ]:
fixture_script = fixtures_dir() / "echo_mcp.py"
subprocess.run(
    [
        "uv", "run", "mms", "add", "echo",
        "--config", str(config_path),
        "--command", "uv",
        "--args", f"run python {fixture_script}",
        "--prefix", "echo",
    ],
    check=True,
    capture_output=True,
)
print("Registered echo fixture.")

async with stm_session() as session:
    result = await session.call_tool("echo__say", {"text": "hello observability"})
    print(extract_text(result))

## 3. Built-in observability: `stm_proxy_stats`

The simplest way to check what STM is doing — call the
`stm_proxy_stats` MCP tool from inside a session.

In [ ]:
async with stm_session() as session:
    stats = await session.call_tool("stm_proxy_stats", {})
    print(extract_text(stats))

## 4. Built-in observability: `stm_proxy_health`

Check upstream connectivity and circuit-breaker state.

In [ ]:
async with stm_session() as session:
    health = await session.call_tool("stm_proxy_health", {})
    print(extract_text(health))

## 5. What Langfuse tracing adds

The MCP tools give you a snapshot; SQLite gives you history.
Langfuse adds **live, nested span visualization** across your
entire proxy pipeline:

| Span name | When | Metadata |
|---|---|---|
| `proxy_call` | Every proxy invocation | `server`, `tool`, `trace_id` |
| `proxy_call_clean` | Content cleaning | `server`, `tool` |
| `proxy_call_compress` | Compression | `server`, `tool`, `strategy` |
| `proxy_call_surface` | Memory injection | `server`, `tool` |
| `proxy_call_index` | Auto-indexing | `server`, `tool` |
| `stm_surfacing_feedback` | Feedback tool | `surfacing_id`, `rating` |
| `stm_surfacing_stats` | Stats query | `tool` |

Every span is a no-op `nullcontext` when Langfuse is disabled
— zero overhead.

## 6. Enabling Langfuse

Set these environment variables before starting the STM server:

```bash
export MEMTOMEM_STM_LANGFUSE__ENABLED=true
export MEMTOMEM_STM_LANGFUSE__PUBLIC_KEY=pk-lf-...
export MEMTOMEM_STM_LANGFUSE__SECRET_KEY=sk-lf-...
export MEMTOMEM_STM_LANGFUSE__HOST=https://cloud.langfuse.com
```

For self-hosted Langfuse, point `HOST` at `http://localhost:3000`.

In [ ]:
_HAS_LANGFUSE_KEY = bool(
    os.environ.get("LANGFUSE_PUBLIC_KEY")
    or os.environ.get("MEMTOMEM_STM_LANGFUSE__PUBLIC_KEY")
)
if _HAS_LANGFUSE_KEY:
    print("Langfuse credentials detected — live tracing cells will run.")
else:
    print("No Langfuse credentials. Live tracing cells will be skipped.")
    print("Set MEMTOMEM_STM_LANGFUSE__PUBLIC_KEY to enable.")

## 7. Sampling configuration

For high-throughput deployments, trace only a fraction of calls:

```bash
export MEMTOMEM_STM_LANGFUSE__SAMPLING_RATE=0.1  # trace 10%
```

Default is `1.0` (trace all). SQLite metrics are always
recorded regardless of sampling — they are never skipped.

## 8. Trace context propagation

When STM forwards calls to upstream servers, it includes a
`_trace_id` field in the tool arguments. Upstream servers can
extract this to correlate their own spans with the originating
STM trace. The same mechanism works for LTM searches via the
`McpClientSearchAdapter`.

This makes end-to-end distributed tracing possible across the
full `agent → STM proxy → upstream server` pipeline.

## 9. Log level configuration

STM uses Python's `logging` module. Control verbosity via
environment variable:

```bash
export MEMTOMEM_STM_LOG_LEVEL=WARNING   # default
export MEMTOMEM_STM_LOG_LEVEL=DEBUG     # full pipeline tracing
```

| Level | What it shows |
|-------|---------------|
| `DEBUG` | Pipeline tracing, strategy selection, cache decisions |
| `INFO` | Surfacing injections, config reloads, extraction completions |
| `WARNING` | Failure guards (F1/S1), fallback activations, skips |
| `ERROR` | Unrecoverable failures (rare — most errors fall back) |

The level is read once at startup — restart the server to
apply changes. See
[`docs/operations.md → Logging`](../docs/operations.md#logging)
for format details.

## Recap

| Layer | What you get | When to use |
|---|---|---|
| MCP tools | Live snapshot, agent-accessible | Quick checks |
| SQLite | Full history, offline analysis | Debugging, tuning |
| Logging | `MEMTOMEM_STM_LOG_LEVEL` control | Failure diagnosis (F1/S1) |
| Langfuse | Nested spans, team-shared UI | Production monitoring |

**Where to next:**

- [`docs/operations.md`](../docs/operations.md) — full
  observability reference (logging, spans, data storage, safety)
- [`docs/configuration.md`](../docs/configuration.md) — env
  vars for log level, Langfuse, and all other settings
- [Langfuse docs](https://langfuse.com/docs) — dashboard
  setup and SDK reference